This script will summarize population sizes by state in the ACS and AFC

In [ ]:
import pandas as pd
import zipfile
import ast
import re
import json
import numpy as np
import statistics
import datetime
import matplotlib.pyplot as plt
import gzip
import os
import math

## Get summary of population sizes by state

In [ ]:
with gzip.open(
    '/share/pi/deho-pi/AFC/BQ/GeneratedPatientBaseline_0724.csv.gz', 'rt'
) as f:
    df = pd.read_csv(f, usecols=['fips_state'])

afc_summary = df['fips_state'].value_counts(dropna=False)

#print(afc_summary)

In [ ]:
import censusdata

YEAR = 2024  # use most recent available ACS 1-year

acs_pop = censusdata.download(
    'acs1',
    YEAR,
    censusdata.censusgeo([('state', '*')]),
    ['B01003_001E']  # Total population
)

acs_pop = (
    acs_pop
    .reset_index()
    .assign(
        state_fips=lambda d: d['index'].apply(
            lambda x: x.geo[0][1]
        )
    )
    .rename(columns={'B01003_001E': 'acs_population',
                    'state_fips': 'fips_state'})
    [['fips_state', 'acs_population']]
)


In [ ]:
afc_summary = pd.DataFrame(afc_summary)

In [ ]:
afc_summary['fips_state'] = afc_summary.index
afc_summary.index.name = 'ind'

In [ ]:
afc_summary['fips_state'] = (
    afc_summary['fips_state']
        .astype('Int64')
        .astype(str)
        .str.zfill(2)
)
afc_summary = afc_summary.rename(columns={'count': 'afc_population'})

In [ ]:
# pull in child politics data, restrict to the potential children
children_merged = pd.read_csv(
    '/share/pi/deho-pi/AFC/mortonc/processed/patients_ruca_svi.csv.zip',
    low_memory=False
)

child_ids = np.load('/share/pi/deho-pi/AFC/child_party.npy', allow_pickle = True)

In [ ]:
child_ids = pd.DataFrame(child_ids)
child_ids.columns = ['patientuid', 'age', 'household_id', 'party']

In [ ]:
children_merged = children_merged[children_merged['patientuid'].isin(child_ids['patientuid'])]

In [ ]:
len(children_merged), len(child_ids)

In [ ]:
children_merged = children_merged[children_merged['party'].notna()]

In [ ]:
len(children_merged)

In [ ]:
sum(pd.isna(children_merged['fips_state']))

In [ ]:
children_merged = children_merged['fips_state'].value_counts()

In [ ]:
sum(children_merged)

In [ ]:
children_merged = pd.DataFrame(children_merged)
children_merged['fips_state'] = children_merged.index
children_merged.index.name = 'ind'
children_merged['fips_state'] = (
    children_merged['fips_state']
        .astype('Int64')
        .astype(str)
        .str.zfill(2)
)
children_merged = children_merged.rename(columns={'count': 'children_merged_population'})

In [ ]:
merged = pd.merge(pd.DataFrame(acs_pop), pd.DataFrame(afc_summary))

In [ ]:
merged = pd.merge(merged, children_merged)

In [ ]:
sum(children_merged['children_merged_population'])

In [ ]:
merged

In [ ]:
def save_zip_csv(filepath, dataset):
    # write to CSV
    csv_filename = filepath
    dataset.to_csv(csv_filename, index=False)

    # zip CSV
    zip_filename = csv_filename + '.zip'

    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(csv_filename, os.path.basename(csv_filename))

    # remove large csv
    os.remove(csv_filename)
    
    print("Saved!")

In [ ]:
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/processed/state_counts_01202026.csv', merged)